# Solution A3 · Target encoding from scratch: smoothing and out-of-fold encoding

**Type**: Implement from scratch

## Objective

Implement target encoding with smoothing, in a naive version and in an out-of-fold version, and measure how much the naive version
leaks the target: on a column of **pure noise** it should be worth 0.5 of AUC.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, train_test_split

df = ds.loans(with_missing=False)
y = df["defaulted"].to_numpy()
occ = df["occupation"].to_numpy()
noise = np.array([f"r{i}" for i in np.random.default_rng(1).integers(0, 300, len(df))])   # no link to y
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.25, random_state=0)
y_tr, y_te = y[idx_tr], y[idx_te]

## Commented solution

In [ ]:
def te_estimate(cat, y, m):
    d = pd.DataFrame({"c": cat, "y": y.astype(float)})
    g = d.groupby("c")["y"].agg(["sum", "count"])
    overall = d["y"].mean()
    mapping = ((g["sum"] + m * overall) / (g["count"] + m)).to_dict()
    return mapping, overall


def te_apply(cat, mapping, overall):
    return pd.Series(cat).map(mapping).fillna(overall).to_numpy(dtype=float)


def te_oof(cat, y, m, k=5, seed=0):
    out = np.zeros(len(cat))
    for tr, va in KFold(k, shuffle=True, random_state=seed).split(cat):
        mapping, overall = te_estimate(cat[tr], y[tr], m)             # statistics computed WITHOUT the fold `va`
        out[va] = te_apply(cat[va], mapping, overall)
    return out

In [ ]:
m = 5
rows = {}
for name, col in [("occupation", occ), ("noise", noise)]:
    c_tr, c_te = col[idx_tr], col[idx_te]
    mapping, overall = te_estimate(c_tr, y_tr, m)
    naive_tr = te_apply(c_tr, mapping, overall)                  # the training set encoded with its own labels
    oof_tr = te_oof(c_tr, y_tr, m)
    te = te_apply(c_te, mapping, overall)
    rows[name] = {"naive_train_auc": roc_auc_score(y_tr, naive_tr), "oof_train_auc": roc_auc_score(y_tr, oof_tr),
                  "test_auc": roc_auc_score(y_te, te)}
table = pd.DataFrame(rows).T
table.round(3)

## Check

In [ ]:
with verify():
    mapping, glob = te_estimate(np.array(["a", "a", "b"]), np.array([1, 0, 1]), m=1)
    check(close(glob, 2 / 3) and close(mapping["a"], (1 + 2 / 3) / 3) and close(mapping["b"], (1 + 2 / 3) / 2),
          "te_estimate is correct on a hand example", "with m=1: a -> (1 + 2/3)/3, b -> (1 + 2/3)/2")
    check(close(te_apply(np.array(["a", "z"]), mapping, glob), [mapping["a"], glob]),
          "te_apply uses the global mean for unknown categories", "a category not in the mapping gets `overall`")
    c_toy = np.array(list("aabbabab") * 5)
    y_toy = np.tile([1, 0, 0, 1, 1, 1, 0, 0], 5)
    enc = te_oof(c_toy, y_toy, m=2, k=5, seed=0)
    check(enc.shape == (40,) and not np.isnan(enc).any(), "te_oof returns one value per row", "one value for every row")
    check(not np.allclose(enc, te_apply(c_toy, *te_estimate(c_toy, y_toy, 2))),
          "the out-of-fold encoding does NOT match the naive one", "every row must be encoded without itself")
    check(table.loc["noise", "naive_train_auc"] > 0.75,
          "on pure noise the naive encoding seems to predict well (high training AUC): it is leakage",
          "naive_train_auc of the noise should be high: are you encoding the training set with the mapping computed on the training set?")
    check(abs(table.loc["noise", "oof_train_auc"] - 0.5) < 0.08,
          "on pure noise the out-of-fold encoding gives a training AUC close to 0.5",
          "with the out-of-fold encoding the noise should have an AUC of about 0.5")
    check(table.loc["noise", "test_auc"] < 0.65, "on the test set the noise predicts nothing",
          "the test AUC of the noise must stay close to 0.5")
    check(table.loc["occupation", "naive_train_auc"] - table.loc["occupation", "oof_train_auc"] > 0.06,
          "even on a real variable the naive encoding is inflated", "the naive training AUC must exceed the out-of-fold one by at least 0.06")
done()

## What you learned

- On pure noise, which by construction has no link to the target, the naive encoding gets a training AUC of
  about 0.85: the downstream model would "learn" to trust a column that contains nothing. The out-of-fold encoding
  brings the value back to about 0.5, and on the test set the noise is worth about 0.5-0.6.
- Even on the real variable (`occupation`) the naive version is inflated (about 0.67 against 0.57 out-of-fold on the
  training set): the downstream model would trust this column too much and generalize worse.
- *Smoothing* (parameter `m`) is the second defense: a category with few examples has an unreliable mean, and the
  formula pulls it toward the global mean. It does not replace the out-of-fold: it goes alongside it.

## Common pitfall

Applying target encoding with `fit_transform` on the whole dataset before the split or the cross-validation. Even with
smoothing, every row carries with it a part of its own target. It always goes inside the pipeline, with a transformer that
uses the out-of-fold encoding during the fit (you will write it in exercise E3).

## Interview question

> *Click on a question to show its answer.*

<details><summary>What is target leakage in target encoding, and how do you avoid it?</summary>

If the encoding of a row uses the mean of the target of its category computed including the row itself, the encoded
value contains its target: the smaller the category, the stronger the effect (with a single row, the encoding is the
target). You avoid it with out-of-fold encoding (every row encoded with statistics computed without it), with
smoothing toward the global mean for rare categories, and by keeping everything inside the cross-validation.

</details>